# Capítulo 5 · Actividad: ¿quién sobrevivió al Titanic?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_05/01_actividad_titanic_arbol.ipynb)

Entrena un **árbol de decisión** que prediga si un pasajero sobrevivió (`survived`) y consigue el mejor modelo que puedas. Haz todo el proceso que vimos en clase; el preprocesamiento lo decides tú.

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos: `uv run python scripts/download_data.py`

**Versión final: 9 de noviembre de 2026.**

Esta actividad continúa el proyecto 1 del Titanic. Allá dejamos los datos limpios y preparados; aquí usamos esas mismas decisiones para entrenar un árbol de decisión y, sobre todo, para entender cómo llega a sus respuestas.

Lo que mantenemos del proyecto 1:

- La variable objetivo es `survived` (1 sobrevivió, 0 no). Es un problema de clasificación.
- Quitamos las mismas columnas: `alive` (da la respuesta), `class` y `embark_town` (repiten `pclass` y `embarked`), `who`, `adult_male` y `alone` (resumen de sexo, edad y familia) y `deck` (le faltaba el 77%).
- Conservamos las filas repetidas y los valores extremos.
- Separamos 80/20 con `stratify` y `random_state=42`, así que train y test son las mismas filas que en el proyecto 1.
- Los faltantes se rellenan con valores calculados solo con train.

Lo que cambia al usar un árbol:

- No hace falta escalar ni aplicar logaritmo. Lo comprobamos más abajo.
- Las variables nuevas no se aceptan por intuición: se comparan con validación cruzada y nos quedamos con las que ayudan.
- Antes de usar `DecisionTreeClassifier`, calculamos a mano cómo elige sus preguntas y comparamos nuestro resultado con el de scikit-learn.
- El test se usa una sola vez para evaluar el modelo final. La sección 14 repite el proceso con otros repartos solo para medir cuánto cambia el resultado.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import (GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_val_score,
                                     cross_validate, learning_curve, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

# Que scikit-learn devuelva DataFrames (con nombres de columna)
set_config(transform_output="pandas")
sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

df = pd.read_csv(DATA_DIR / "titanic.csv")
df.head()

## 1. Lo que traemos del proyecto 1

Guardamos una copia de los datos originales y quitamos las columnas que ya habíamos decidido quitar.

In [ ]:
df_original = df.copy()

columnas_eliminar = ["alive", "class", "embark_town", "who", "adult_male", "deck", "alone"]
df_limpio = df_original.drop(columns=columnas_eliminar)

print("Filas y columnas:", df_limpio.shape)
df_limpio.isna().sum()

Quedan 891 pasajeros y 8 columnas: `survived` y 7 de entrada. Solo faltan 177 edades y 2 puertos. No los rellenamos aquí: lo hará el Pipeline con valores de train.

In [ ]:
print("Porcentaje de supervivientes:")
print((df_limpio["survived"].value_counts(normalize=True) * 100).round(2))

for col in ["sex", "pclass"]:
    print(f"\nSupervivencia (%) por {col}:")
    print((df_limpio.groupby(col)["survived"].mean() * 100).round(2))

Sobrevivió el 38.38%. Un modelo que dijera siempre "no sobrevivió" acertaría el 61.62%, así que esa es la marca mínima. Como en el proyecto 1, las diferencias más grandes están en el sexo (74.20% de las mujeres contra 18.89% de los hombres) y en la clase (62.96% en primera contra 24.24% en tercera).

## 2. Variables nuevas, cada una con su razón

En el proyecto 1 creamos `family_size`, `age_group` y `edad_faltante`. Aquí las repetimos y agregamos dos más, `fare_por_persona` y `es_mujer_o_nino`. Cada una tiene que responder a algo que sabemos del barco y se valida contra `survived` usando solo train. En la sección 4 las ponemos a competir con otras candidatas.

- `family_size = sibsp + parch + 1`: personas del grupo familiar, contando al pasajero. En el proyecto 1 vimos que viajar solo o en un grupo muy grande bajaba la supervivencia. Un árbol puede dividir `sibsp` y `parch` por separado, pero no sumarlos, así que le damos la suma ya hecha.
- `fare_por_persona = fare / family_size`: `fare` es el precio del boleto y un boleto cubre a todo el grupo. Eso se ve en los datos (celda de abajo). Dividir entre el tamaño del grupo da lo que pagó cada persona, que se parece más a su nivel económico. Es una aproximación, porque el grupo de un boleto no siempre coincide con la familia registrada.
- `es_mujer_o_nino`: 1 si el pasajero es mujer o tiene 12 años o menos. En el hundimiento se siguió, en general, la norma de que las mujeres y los niños abordaran primero los botes, así que esta variable resume esa norma. Si la edad falta, el pasajero cuenta como no niño (para una mujer da igual, porque ya cumple por ser mujer).
- `edad_faltante`: 1 si la edad no estaba registrada. Es la misma idea del proyecto 1.
- `age_group`: tramos de edad (Infancia hasta 12 años, Adolescencia hasta 17, Adultez hasta 59 y Mayor). Es la misma agrupación del proyecto 1.

Primero la evidencia de que `fare` es el precio de todo el boleto: los pasajeros con 8 hermanos o cónyuges son de una sola familia grande.

In [ ]:
familia_grande = df_limpio[df_limpio["sibsp"] == 8]
print("Pasajeros con sibsp = 8:", len(familia_grande))
print("Valores distintos de fare entre ellos:", familia_grande["fare"].unique())
familia_grande[["pclass", "sibsp", "parch", "fare"]]

In [ ]:
df_limpio["family_size"] = df_limpio["sibsp"] + df_limpio["parch"] + 1
df_limpio["fare_por_persona"] = df_limpio["fare"] / df_limpio["family_size"]
df_limpio["es_mujer_o_nino"] = ((df_limpio["sex"] == "female") | (df_limpio["age"] <= 12)).astype(int)
df_limpio["edad_faltante"] = df_limpio["age"].isna().astype(int)
df_limpio["age_group"] = pd.cut(
    df_limpio["age"], bins=[-1, 12, 17, 59, np.inf],
    labels=["Infancia", "Adolescencia", "Adultez", "Mayor"]
)

df_limpio[["sex", "age", "sibsp", "parch", "family_size", "fare", "fare_por_persona", "es_mujer_o_nino", "edad_faltante", "age_group"]].head(8)

Los 7 pasajeros con `sibsp = 8` tienen la misma clase, los mismos padres o hijos y la misma tarifa (69.55): es un solo boleto, no 7 precios iguales. Por eso `fare_por_persona` tiene sentido. En `age_group`, las edades que faltan quedan vacías; el Pipeline las rellenará después con el grupo más frecuente, que en train es Adultez. Es lo mismo que pasaba en el proyecto 1, donde la mediana (28.5) caía en Adultez.

### Separar train y test

Separamos igual que en el proyecto 1. Las variables nuevas se calculan fila por fila y no aprenden nada de los datos, así que crearlas antes de separar no filtra información del test.

In [ ]:
X = df_limpio.drop(columns="survived")
y = df_limpio["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Supervivencia en train (%):", round(y_train.mean() * 100, 2))
print("Supervivencia en test (%):", round(y_test.mean() * 100, 2))

Quedan 712 pasajeros en train y 179 en test, igual que en el proyecto 1, con 38.34% y 38.55% de supervivientes.

### Validar cada variable contra `survived` (solo train)

Para `fare_por_persona` hacemos grupos solo para mirar; el árbol usará el valor continuo.

In [ ]:
df_train = X_train.copy()
df_train["survived"] = y_train
df_train["grupo_fare_por_persona"] = pd.cut(
    df_train["fare_por_persona"], bins=[-1, 7.5, 10, 20, 40, np.inf],
    labels=["Hasta 7.5", "Más de 7.5 a 10", "Más de 10 a 20", "Más de 20 a 40", "Más de 40"]
)

for col in ["family_size", "grupo_fare_por_persona", "es_mujer_o_nino", "edad_faltante", "age_group"]:
    resumen = df_train.groupby(col, observed=True)["survived"].agg(["mean", "count"])
    resumen["mean"] = (resumen["mean"] * 100).round(2)
    resumen = resumen.rename(columns={"mean": "supervivencia (%)", "count": "pasajeros"})
    display(resumen)

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(resumen.index.astype(str), resumen["supervivencia (%)"])
    ax.set_title(f"Supervivencia por {col} en train")
    ax.set_xlabel(col)
    ax.set_ylabel("Supervivencia (%)")
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.show()

Lo que muestran las tablas, solo con train:

- `family_size`: quien viaja solo sobrevive el 29.95%. En grupos de 2, 3 y 4 personas sube a 55.04%, 62.34% y 72.73%. Con 5 o más personas baja otra vez (21.43% con 5 y 6.25% con 6), pero esos grupos tienen entre 4 y 16 pasajeros, así que sus porcentajes cambian mucho con pocos casos.
- `fare_por_persona`: la supervivencia sube cuando sube lo que pagó cada persona, de 25.77% (hasta 7.5 libras) a 68.83% (más de 40). Tiene sentido, porque lo que paga cada persona está ligado a su clase.
- `es_mujer_o_nino`: sobrevivió el 72.60% de quienes cumplen la regla (281 pasajeros) y el 16.01% de quienes no (431). Es la diferencia más grande de todas las variables.
- `edad_faltante`: sobrevivió el 40.52% de quienes tenían la edad registrada y el 29.20% de quienes no.
- `age_group`: los niños de hasta 12 años sobrevivieron más (60.00%) y los mayores de 59 menos (16.67%, con solo 18 pasajeros). Estos porcentajes no incluyen a los 137 pasajeros de train sin edad.

Las cinco variables se relacionan con `survived`, pero eso todavía no dice si le sirven a un árbol. Eso lo comprobamos en la sección 4.

## 3. Preprocesamiento dentro de un Pipeline

En el proyecto 1 rellenamos a mano y después ajustamos el `ColumnTransformer`. Ahora todo va dentro del Pipeline del modelo. Así, cuando la validación cruzada separa train en partes, cada parte aprende su propia mediana y su propia moda, y nunca usa las filas que está evaluando.

La función `armar_preprocesador` junta los tres tipos de columna que usamos en el proyecto 1:

- Numéricas: imputador con la mediana. No las escalamos.
- Nominales (sin orden): imputador con el valor más frecuente y one-hot con `drop="if_binary"`, que deja una sola columna cuando hay dos categorías.
- Ordinales (con orden): imputador con el valor más frecuente y `OrdinalEncoder` con el orden escrito de forma explícita.

`pclass` ya viene como 1, 2 y 3, que es su orden, así que la tratamos como numérica. Así los cortes del árbol se leen directamente ("clase menor o igual a 1.5").

In [ ]:
ORDEN = {"age_group": ["Infancia", "Adolescencia", "Adultez", "Mayor"]}


def armar_preprocesador(numericas, nominales=(), ordinales=()):
    pasos = [("num", SimpleImputer(strategy="median"), list(numericas))]
    if nominales:
        pasos.append(("nom", Pipeline([
            ("imputador", SimpleImputer(strategy="most_frequent")),
            ("codificador", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False)),
        ]), list(nominales)))
    if ordinales:
        pasos.append(("ord", Pipeline([
            ("imputador", SimpleImputer(strategy="most_frequent")),
            ("codificador", OrdinalEncoder(categories=[ORDEN[c] for c in ordinales])),
        ]), list(ordinales)))
    return ColumnTransformer(pasos)


# Prueba con todas las variables, solo para ver qué entrega
prueba = armar_preprocesador(
    numericas=["age", "pclass", "sibsp", "parch", "fare", "family_size", "fare_por_persona", "es_mujer_o_nino"],
    nominales=["sex", "embarked", "edad_faltante"],
    ordinales=["age_group"],
)
X_prueba = prueba.fit_transform(X_train)
print("Columnas:", X_prueba.shape[1], "| faltantes:", X_prueba.isna().sum().sum())
X_prueba.head()

## 4. ¿Qué variables conservar?

Probamos varios conjuntos de variables con tres árboles distintos: uno sin límites y dos con límites de profundidad y de tamaño de hoja (la hoja es el grupo final donde el árbol da su respuesta). Cada combinación se evalúa con validación cruzada de 5 partes repetida 10 veces (50 evaluaciones) y solo con train. Repetimos porque con 712 pasajeros una sola validación cruzada cambia bastante según cómo caigan los pasajeros en cada parte. Si una diferencia es menor que la desviación (alrededor de 0.03), no la tomamos como real.

Conjuntos que comparamos:

- A: las columnas originales (`age`, `pclass`, `sibsp`, `parch`, `fare`, `sex`, `embarked`).
- B: cambiamos `sibsp`, `parch` y `fare` por `family_size` y `fare_por_persona`.
- C: B sin `embarked`.
- D, E y F: C más `edad_faltante`, `age_group` o `who` (la columna que quitamos en el proyecto 1).
- G, H e I: C quitando una variable a la vez (`family_size`, `fare_por_persona` o `age`), o poniendo `fare` en lugar de `fare_por_persona`.

In [ ]:
# who solo se agrega aquí para comprobar si perdimos algo al quitarla en el proyecto 1
X_train_ext = X_train.join(df_original["who"])

numericas_c = ["age", "pclass", "family_size", "fare_por_persona"]

conjuntos = {
    "A originales": (["age", "pclass", "sibsp", "parch", "fare"], ["sex", "embarked"], []),
    "B family_size y fare_por_persona": (numericas_c, ["sex", "embarked"], []),
    "C = B sin embarked": (numericas_c, ["sex"], []),
    "D = C + edad_faltante": (numericas_c, ["sex", "edad_faltante"], []),
    "E = C + age_group": (numericas_c, ["sex"], ["age_group"]),
    "F = C + who": (numericas_c, ["sex", "who"], []),
    "G = C sin family_size": (["age", "pclass", "fare_por_persona"], ["sex"], []),
    "H = C con fare en lugar de fare_por_persona": (["age", "pclass", "family_size", "fare"], ["sex"], []),
    "I = C sin age": (["pclass", "family_size", "fare_por_persona"], ["sex"], []),
}

arboles = {
    "sin límites": {},
    "profundidad 4, hojas de 5+": {"max_depth": 4, "min_samples_leaf": 5},
    "profundidad 5, hojas de 8+": {"max_depth": 5, "min_samples_leaf": 8},
}

cv_repetida = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEED)

def comparar_conjuntos(conjuntos, datos):
    filas = []
    for nombre_conjunto, (num, nom, ordi) in conjuntos.items():
        fila = {"conjunto": nombre_conjunto}
        for nombre_arbol, parametros in arboles.items():
            modelo = Pipeline([
                ("preprocesador", armar_preprocesador(num, nom, ordi)),
                ("arbol", DecisionTreeClassifier(random_state=SEED, **parametros)),
            ])
            scores = cross_val_score(modelo, datos, y_train, cv=cv_repetida, scoring="accuracy")
            fila[nombre_arbol] = f"{scores.mean():.3f} ± {scores.std():.3f}"
        filas.append(fila)
    return pd.DataFrame(filas).set_index("conjunto")


comparar_conjuntos(conjuntos, X_train_ext)

Cada celda es la accuracy media ± la desviación entre las 50 evaluaciones.

- **Sin límites, el árbol casi no distingue entre conjuntos.** Todos quedan entre 0.762 y 0.797, porque se memoriza el train sin importar qué variables reciba. La única diferencia llamativa es la de I (sin `age`): 0.797.
- **Con límites, C es mejor que A.** Con profundidad 4 sube de 0.815 a 0.829 y con profundidad 5 de 0.809 a 0.829. Son 1 o 2 puntos, menos que la desviación, pero aparecen en los dos árboles. Es una mejora pequeña, no una prueba.
- **`fare_por_persona` aporta.** H, que usa `fare` en lugar de `fare_por_persona`, baja a 0.812 en los dos árboles limitados.
- **`family_size` aporta.** G (C sin `family_size`) baja a 0.817 y 0.823.
- **`edad_faltante`, `age_group` y `who` no cambian nada.** D, E y F quedan entre 0.827 y 0.834, dentro del ruido. Quitar `who` en el proyecto 1 no nos costó nada.
- **`embarked` sobra.** B, con `embarked`, da 0.824 y 0.821; C, sin ella, 0.829 en los dos. Es coherente con el proyecto 1, donde vimos que la diferencia por puerto se explicaba con la clase.
- **`age` se queda por ahora.** I (sin `age`) baja a 0.816 y 0.799 con límites. Más abajo vemos que una variable nueva la puede reemplazar.

### Otras variables candidatas

Hasta aquí probamos variables del proyecto 1. Ahora probamos otras con una razón del hundimiento, comparándolas contra el conjunto C y contra las que ya teníamos:

- `es_mujer_o_nino`: la norma de abordaje (mujeres y niños primero).
- `es_nino`: 12 años o menos.
- `familia_pequena`: grupo de 2 a 4 personas, que es el rango con más supervivencia.
- `viaja_solo`: grupo de una persona.
- `cubierta_conocida`: si se conoce la cubierta del camarote (`deck`).
- `mujer_clase_alta`: mujer de primera o segunda clase.
- `madre_probable`: mujer de 18 años o más con padres o hijos a bordo (`parch` mayor que 0). Es una aproximación.

También probamos reemplazar `family_size` por `familia_pequena` o por `viaja_solo`, que parecen más fáciles de justificar, para ver si de verdad son mejores.

In [ ]:
# Se calculan fila por fila con columnas que ya teníamos, así que no aprenden nada de los datos
mujer = df_original["sex"] == "female"
candidatas = pd.DataFrame({
    "es_nino": (df_original["age"] <= 12).astype(int),
    "familia_pequena": df_limpio["family_size"].between(2, 4).astype(int),
    "viaja_solo": (df_limpio["family_size"] == 1).astype(int),
    "cubierta_conocida": df_original["deck"].notna().astype(int),
    "mujer_clase_alta": (mujer & (df_original["pclass"] <= 2)).astype(int),
    "madre_probable": (mujer & (df_original["age"] >= 18) & (df_original["parch"] > 0)).astype(int),
})
X_train_cand = X_train.join(candidatas)

sin_family = ["age", "pclass", "fare_por_persona"]
numericas_j = numericas_c + ["es_mujer_o_nino"]
sin_age = [c for c in numericas_j if c != "age"]
conjuntos_candidatas = {
    "C (referencia)": (numericas_c, ["sex"], []),
    "J = C + es_mujer_o_nino": (numericas_j, ["sex"], []),
    "J sin age": (sin_age, ["sex"], []),
    "J sin sex": (numericas_j, [], []),
    "K = J sin age ni sex (4 variables)": (sin_age, [], []),
    "C + es_nino": (numericas_c + ["es_nino"], ["sex"], []),
    "C + familia_pequena": (numericas_c + ["familia_pequena"], ["sex"], []),
    "C + viaja_solo": (numericas_c + ["viaja_solo"], ["sex"], []),
    "C + cubierta_conocida": (numericas_c + ["cubierta_conocida"], ["sex"], []),
    "C + mujer_clase_alta": (numericas_c + ["mujer_clase_alta"], ["sex"], []),
    "C + madre_probable": (numericas_c + ["madre_probable"], ["sex"], []),
    "C sin family_size": (sin_family, ["sex"], []),
    "C con familia_pequena en vez de family_size": (sin_family + ["familia_pequena"], ["sex"], []),
    "C con viaja_solo en vez de family_size": (sin_family + ["viaja_solo"], ["sex"], []),
}

comparar_conjuntos(conjuntos_candidatas, X_train_cand)

La referencia es C (0.829 con profundidad 4 y 0.829 con profundidad 5). Lo que muestra la tabla:

- **`es_mujer_o_nino` es la mejor candidata.** J (C más esta variable) sube a 0.835 en los dos árboles limitados. Es la única de las siete que mejora de forma pareja, y tiene una razón de fondo: es la norma que se siguió, en general, para abordar los botes.
- **Con `es_mujer_o_nino`, sobran `age` y `sex`.** Esa variable ya resume lo que importaba de las dos: un árbol no puede combinar sexo y edad en una sola pregunta, pero con esta variable sí. Quitarlas deja el resultado igual o mejor. K, con solo 4 variables, saca 0.841 con profundidad 4 y 0.828 con profundidad 5, y sin límites sube de 0.774 a 0.820. En promedio de los tres árboles, K da 0.830 contra 0.815 de J. Con menos columnas, el árbol tiene menos cortes sin sentido para memorizar.
- **`family_size` se queda.** Quitarla baja el resultado a 0.817 y 0.823. Y las dos variables que parecían más fáciles de justificar, `familia_pequena` (de 2 a 4 personas) y `viaja_solo`, funcionan peor que ella cuando la reemplazan (0.817 y 0.815 con profundidad 4). La razón está en la sección 2: el efecto no es lineal, y la cifra completa le da al árbol más información que una sola marca.

In [ ]:
# Sobrevivientes entre quienes cumplen cada condición y entre quienes no (solo train)
datos_candidatas = X_train_cand.copy()
datos_candidatas["survived"] = y_train
columnas_candidatas = ["es_mujer_o_nino", "es_nino", "familia_pequena", "viaja_solo", "cubierta_conocida", "mujer_clase_alta", "madre_probable"]

resumen_candidatas = pd.DataFrame({col: datos_candidatas.groupby(col)["survived"].mean() * 100 for col in columnas_candidatas}).T
resumen_candidatas.columns = ["no cumple (%)", "cumple (%)"]
resumen_candidatas["pasajeros que cumplen"] = [int(datos_candidatas[col].sum()) for col in resumen_candidatas.index]
display(resumen_candidatas.round(1))

print("Cubierta conocida según la clase (pasajeros de train):")
pd.crosstab(datos_candidatas["pclass"], datos_candidatas["cubierta_conocida"], colnames=["cubierta conocida"])

- **`cubierta_conocida` engaña.** Tiene una diferencia enorme de supervivencia (70.4% contra 29.1%), pero baja el resultado (0.822 con profundidad 4). La tabla de arriba explica por qué: de 171 pasajeros de primera clase en train se conoce la cubierta de 135, y de 391 de tercera solo de 9. Es casi una copia de `pclass`. Además, ese dato pudo registrarse después del hundimiento, así que no lo usamos.
- **`mujer_clase_alta`** (94.9% de supervivencia) tampoco ayuda (0.821 con profundidad 4): el árbol ya llega solo a esa combinación. `es_nino`, `viaja_solo`, `familia_pequena` y `madre_probable` no cambian el resultado.

Decisión: nos quedamos con K, es decir, 4 variables: `es_mujer_o_nino`, `pclass`, `fare_por_persona` y `family_size`. Cada una responde a algo distinto (la norma de abordaje, la clase, el nivel económico y el tamaño del grupo) y ninguna repite a otra. Como elegimos con validación cruzada sobre train, los números son algo optimistas. El test lo revisamos hasta el final.

In [ ]:
numericas = ["pclass", "family_size", "fare_por_persona", "es_mujer_o_nino"]
nominales = []


def armar_modelo(**parametros_arbol):
    """Preprocesamiento del conjunto elegido y un árbol con los parámetros que se le pasen."""
    return Pipeline([
        ("preprocesador", armar_preprocesador(numericas, nominales)),
        ("arbol", DecisionTreeClassifier(random_state=SEED, **parametros_arbol)),
    ])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

### ¿Hace falta escalar o aplicar logaritmo?

En el proyecto 1 escalamos con `StandardScaler` y aplicamos `log1p` a `fare`. Un árbol solo pregunta "¿es mayor que este umbral?", y el escalado y el logaritmo no cambian el orden de los pasajeros. Lo comprobamos con las variables elegidas.

In [ ]:
con_transformacion = ColumnTransformer([
    ("num", Pipeline([
        ("imputador", SimpleImputer(strategy="median")),
        ("escalador", StandardScaler()),
    ]), ["pclass", "family_size", "es_mujer_o_nino"]),
    ("tarifa", Pipeline([
        ("imputador", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("escalador", StandardScaler()),
    ]), ["fare_por_persona"]),
])

for nombre_arbol, parametros in arboles.items():
    sin = cross_val_score(armar_modelo(**parametros), X_train, y_train, cv=cv_repetida, scoring="accuracy")
    con = cross_val_score(
        Pipeline([("preprocesador", con_transformacion),
                  ("arbol", DecisionTreeClassifier(random_state=SEED, **parametros))]),
        X_train, y_train, cv=cv_repetida, scoring="accuracy")
    print(f"{nombre_arbol:<28} sin transformar: {sin.mean():.3f} | con log y escalado: {con.mean():.3f}")

Los resultados son idénticos con 3 decimales (0.820, 0.841 y 0.828). El escalado y el logaritmo cambian los números pero no el orden de los pasajeros, y el árbol solo usa el orden. Por eso en este modelo dejamos las variables en sus unidades reales, y los umbrales se leen directo (libras por persona, personas y clase). Si más adelante cambiamos a un modelo sensible a la escala, volvemos a agregar el escalado.

## 5. Cómo elige un árbol sus preguntas

Un árbol de decisión hace preguntas de sí o no sobre las variables. En cada nodo prueba todas las preguntas posibles y se queda con la que deja los dos grupos más **puros**, es decir, con pasajeros de una sola clase. Aquí hacemos ese cálculo a mano y después lo comparamos con scikit-learn.

Usamos los datos de train ya preprocesados (con los mismos pasos del Pipeline).

In [ ]:
preprocesador = armar_preprocesador(numericas, nominales)
X_train_proc = preprocesador.fit_transform(X_train)
y_train_np = y_train.to_numpy()

print("Columnas que ve el árbol:", list(X_train_proc.columns))
X_train_proc.head()

### 5.1 Impureza de un nodo

La impureza mide qué tan mezclado está un grupo. Con `p` como la proporción de sobrevivientes:

- **Gini** = 1 − p² − (1 − p)². Es la probabilidad de equivocarnos si etiquetáramos al azar según las proporciones del grupo.
- **Entropía** = −p·log₂(p) − (1 − p)·log₂(1 − p). Mide el desorden del grupo.

En las dos, 0 significa un grupo puro y el valor más alto aparece cuando hay 50% de cada clase.

In [ ]:
def gini(y):
    p = np.mean(y)  # y vale 0 o 1, así que su promedio es la proporción de sobrevivientes
    return 1 - p**2 - (1 - p)**2


def entropia(y):
    p = np.mean(y)
    if p == 0 or p == 1:
        return 0.0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)


proporciones = np.linspace(0, 1, 101)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(proporciones, [gini(np.array([1] * round(p * 100) + [0] * (100 - round(p * 100)))) for p in proporciones], label="Gini")
ax.plot(proporciones, [entropia(np.array([1] * round(p * 100) + [0] * (100 - round(p * 100)))) for p in proporciones], label="Entropía")
ax.set_title("Impureza según la proporción de sobrevivientes")
ax.set_xlabel("Proporción de sobrevivientes (p)")
ax.set_ylabel("Impureza")
ax.legend()
plt.show()

Las dos curvas valen 0 en los extremos y suben hasta el centro. La entropía llega a 1 y Gini a 0.5, pero ordenan los grupos casi igual. Ahora calculamos la impureza del nodo raíz, que es todo train:

In [ ]:
print("Pasajeros en train:", len(y_train_np))
print("Sobrevivieron:", y_train_np.sum(), "| No sobrevivieron:", (y_train_np == 0).sum())
print("Proporción de sobrevivientes:", round(y_train_np.mean(), 4))
print("Gini de la raíz:", round(gini(y_train_np), 4))
print("Entropía de la raíz:", round(entropia(y_train_np), 4))

# Comprobación: un árbol de una sola pregunta guarda la impureza de su raíz
for criterio, medida in [("gini", gini), ("entropy", entropia)]:
    arbol_de_una_pregunta = DecisionTreeClassifier(criterion=criterio, max_depth=1, random_state=SEED)
    arbol_de_una_pregunta.fit(X_train_proc, y_train)
    print(f"Raíz según scikit-learn ({criterio}):", round(arbol_de_una_pregunta.tree_.impurity[0], 4),
          "| la nuestra:", round(medida(y_train_np), 4))

Coinciden. La raíz está bastante mezclada (Gini 0.47, cerca del máximo de 0.5), así que hay mucho que separar.

### 5.2 Una pregunta a mano

Probamos la pregunta "¿el pasajero es mujer o niño?". En las columnas preprocesadas, `num__es_mujer_o_nino` vale 1 si lo es y 0 si no, así que la pregunta se escribe `num__es_mujer_o_nino <= 0.5`: el lado "sí" (izquierda) son quienes no son mujer ni niño, y el lado "no" (derecha) quienes sí lo son. Para cada lado calculamos su impureza, las ponderamos por el número de pasajeros de cada lado, y la **ganancia** es cuánto baja la impureza respecto al nodo padre.

In [ ]:
es_izquierda = (X_train_proc["num__es_mujer_o_nino"] <= 0.5).to_numpy()
y_izquierda = y_train_np[es_izquierda]    # ni mujeres ni niños
y_derecha = y_train_np[~es_izquierda]     # mujeres o niños

tabla = pd.DataFrame({
    "pasajeros": [len(y_train_np), len(y_izquierda), len(y_derecha)],
    "sobreviven": [y_train_np.sum(), y_izquierda.sum(), y_derecha.sum()],
    "gini": [gini(y_train_np), gini(y_izquierda), gini(y_derecha)],
}, index=["Nodo padre", "Izquierda (ni mujer ni niño)", "Derecha (mujer o niño)"])
display(tabla.round(4))

gini_ponderado = (len(y_izquierda) / len(y_train_np)) * gini(y_izquierda) + (len(y_derecha) / len(y_train_np)) * gini(y_derecha)
print("Gini ponderado de los dos grupos:", round(gini_ponderado, 4))
print("Ganancia (Gini del padre menos Gini ponderado):", round(gini(y_train_np) - gini_ponderado, 4))

Entre quienes no son mujer ni niño sobreviven 69 de 431 (16.0%), y entre las mujeres y los niños 204 de 281 (72.6%), así que los dos grupos son más puros que el padre. La ganancia (0.153) dice cuánto mejoró la pregunta; el árbol prefiere la pregunta con mayor ganancia.

### 5.3 Probar todas las preguntas posibles

Para una variable numérica, las preguntas posibles son "¿valor <= umbral?", con un umbral en el punto medio entre cada par de valores consecutivos que aparecen en los datos. Probamos todos los umbrales de todas las variables y calculamos la ganancia de cada pregunta.

In [ ]:
def impureza_ponderada(y_izquierda, y_derecha, medida):
    n = len(y_izquierda) + len(y_derecha)
    return len(y_izquierda) / n * medida(y_izquierda) + len(y_derecha) / n * medida(y_derecha)


def todas_las_preguntas(X, y, medida=gini):
    """Ganancia de cada pregunta 'columna <= umbral' posible en estos datos."""
    y = np.asarray(y)
    impureza_padre = medida(y)
    filas = []
    for columna in X.columns:
        valores = np.unique(X[columna])
        umbrales = (valores[:-1] + valores[1:]) / 2   # punto medio entre valores consecutivos
        v = X[columna].to_numpy()
        for umbral in umbrales:
            es_izq = v <= umbral
            ganancia = impureza_padre - impureza_ponderada(y[es_izq], y[~es_izq], medida)
            filas.append({"columna": columna, "umbral": umbral, "ganancia": ganancia})
    return pd.DataFrame(filas)


preguntas = todas_las_preguntas(X_train_proc, y_train_np)
print("Preguntas probadas en la raíz:", len(preguntas))

print("\nLas 8 mejores:")
display(preguntas.sort_values("ganancia", ascending=False, kind="stable").head(8).round(4))

print("La mejor de cada columna:")
mejores_por_columna = preguntas.loc[preguntas.groupby("columna")["ganancia"].idxmax()]
display(mejores_por_columna.sort_values("ganancia", ascending=False).round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for columna, ax in zip(["num__pclass", "num__fare_por_persona", "num__family_size"], axes):
    datos = preguntas[preguntas["columna"] == columna]
    ax.plot(datos["umbral"], datos["ganancia"])
    ax.set_title(f"Ganancia de cada umbral en {columna}")
    ax.set_xlabel("Umbral")
    ax.set_ylabel("Ganancia (Gini)")
plt.tight_layout()
plt.show()

En la raíz se probaron 264 preguntas. La mejor es, por mucho, `num__es_mujer_o_nino <= 0.5` (ganancia 0.1530). La siguiente, `num__pclass <= 2.5`, tiene 0.0481, más de tres veces menos, y después viene `num__fare_por_persona <= 26.14` con 0.0471. Es el mismo orden que vimos en el análisis: primero la norma de abordaje, después la clase y lo que pagó cada persona. En `family_size` el mejor umbral es 1.5, que separa a quien viaja solo (ganancia de 0.0220).

En las gráficas, cada punto es un umbral posible. En `fare_por_persona` la ganancia es irregular por debajo de 100 libras, porque muchos umbrales parecidos dan ganancias parecidas. En `pclass` solo hay dos umbrales posibles (1.5 y 2.5), y gana el de 2.5, que separa a tercera clase de las otras dos.

Comprobamos con scikit-learn que elige la misma pregunta:

In [ ]:
raiz_sklearn = DecisionTreeClassifier(max_depth=1, random_state=SEED).fit(X_train_proc, y_train)
columna_sklearn = X_train_proc.columns[raiz_sklearn.tree_.feature[0]]
umbral_sklearn = raiz_sklearn.tree_.threshold[0]

mejor = preguntas.sort_values("ganancia", ascending=False, kind="stable").iloc[0]
print("Pregunta de la raíz según nuestro cálculo:", mejor["columna"], "<=", round(mejor["umbral"], 2))
print("Pregunta de la raíz según scikit-learn:   ", columna_sklearn, "<=", round(umbral_sklearn, 2))

### 5.4 Crecer el árbol completo

Después de la raíz, cada grupo se trata como un nodo nuevo: se vuelven a probar todas las preguntas solo con los pasajeros de ese grupo. Se repite hasta que el grupo es puro o llega a la profundidad máxima. Cada hoja responde con la clase que tiene la mayoría de sus pasajeros.

Lo escribimos como una función que se llama a sí misma (recursiva).

In [ ]:
def construir_arbol(X, y, max_profundidad, medida=gini, profundidad=0):
    y = np.asarray(y)
    nodo = {"n": len(y), "sobreviven": int(y.sum()), "impureza": medida(y)}

    if profundidad == max_profundidad or nodo["impureza"] == 0:
        return nodo                                   # hoja

    preguntas = todas_las_preguntas(X, y, medida)
    if preguntas.empty:
        return nodo
    mejor = preguntas.sort_values("ganancia", ascending=False, kind="stable").iloc[0]
    if mejor["ganancia"] <= 0:
        return nodo

    es_izq = (X[mejor["columna"]] <= mejor["umbral"]).to_numpy()
    nodo["columna"] = mejor["columna"]
    nodo["umbral"] = mejor["umbral"]
    nodo["izquierda"] = construir_arbol(X[es_izq], y[es_izq], max_profundidad, medida, profundidad + 1)
    nodo["derecha"] = construir_arbol(X[~es_izq], y[~es_izq], max_profundidad, medida, profundidad + 1)
    return nodo


def mostrar_arbol(nodo, sangria=""):
    if "columna" not in nodo:
        clase = int(nodo["sobreviven"] / nodo["n"] > 0.5)
        print(f"{sangria}|--- clase {clase} ({nodo['sobreviven']} de {nodo['n']} sobreviven)")
        return
    print(f"{sangria}|--- {nodo['columna']} <= {nodo['umbral']:.2f}")
    mostrar_arbol(nodo["izquierda"], sangria + "|   ")
    print(f"{sangria}|--- {nodo['columna']} >  {nodo['umbral']:.2f}")
    mostrar_arbol(nodo["derecha"], sangria + "|   ")


def predecir_pasajero(nodo, fila):
    while "columna" in nodo:
        nodo = nodo["izquierda"] if fila[nodo["columna"]] <= nodo["umbral"] else nodo["derecha"]
    return int(nodo["sobreviven"] / nodo["n"] > 0.5)


def contar_hojas(nodo):
    if "columna" not in nodo:
        return 1
    return contar_hojas(nodo["izquierda"]) + contar_hojas(nodo["derecha"])


arbol_a_mano = construir_arbol(X_train_proc, y_train_np, max_profundidad=3)
mostrar_arbol(arbol_a_mano)

Este es el árbol de profundidad 3 que construimos con nuestras propias funciones. Se lee de arriba abajo. Por ejemplo, una mujer o un niño de primera o segunda clase que pagó 38.13 libras o menos por persona cae en una hoja con 100 pasajeros, de los que sobreviven 93. Las 47 personas que pagaron más de 38.13 sobreviven todas.

Hay tres cosas para notar:

- En la rama de quienes no son mujer ni niño y pagaron 26.14 libras o menos, la pregunta `fare_por_persona <= 3.88` separa a 25 pasajeros, de los que no sobrevive ninguno, de 321, de los que sobreviven 33 (10.3%). Las dos hojas predicen "no sobrevive", así que esa pregunta no cambia ninguna predicción. Aun así hace los grupos más puros. Un árbol puede hacer preguntas que bajan la impureza sin cambiar la clase que predice, y eso mejora sus probabilidades. Por eso ayuda más con el AUC (que mide qué tan bien el modelo ordena a quienes sobreviven por encima de quienes no) que con la accuracy.
- Las mujeres y los niños de tercera clase quedan divididos por el tamaño del grupo: con 4 personas o menos sobreviven 60 de 99 (60.6%), y con 5 o más solo 4 de 35 (11.4%). Es una de las razones por las que `family_size` aporta.
- Entre quienes no son mujer ni niño, la rama de tarifa entre 26.14 y 30.59 libras por persona predice "sobrevive" (21 de 33 sobreviven, 63.6%), y quienes pagaron menos o más predicen "no sobrevive". Un grupo de 33 pasajeros con una ventana tan estrecha de tarifa es poco confiable. Más adelante vemos qué pasa con esa regla en test.

Ahora comparamos con scikit-learn en varias profundidades. Medimos cuántas hojas tiene cada árbol y qué porcentaje de las 712 predicciones de train coincide.

In [ ]:
filas = []
for profundidad in [1, 2, 3, 4, 5]:
    nuestro = construir_arbol(X_train_proc, y_train_np, max_profundidad=profundidad)
    pred_nuestra = np.array([predecir_pasajero(nuestro, fila) for _, fila in X_train_proc.iterrows()])

    de_sklearn = DecisionTreeClassifier(max_depth=profundidad, random_state=SEED).fit(X_train_proc, y_train)
    pred_sklearn = de_sklearn.predict(X_train_proc)

    filas.append({
        "profundidad": profundidad,
        "hojas (nuestro)": contar_hojas(nuestro),
        "hojas (sklearn)": de_sklearn.get_n_leaves(),
        "accuracy train (nuestro)": (pred_nuestra == y_train_np).mean(),
        "accuracy train (sklearn)": (pred_sklearn == y_train_np).mean(),
        "predicciones iguales (%)": (pred_nuestra == pred_sklearn).mean() * 100,
    })

pd.DataFrame(filas).set_index("profundidad").round(4)

Nuestro árbol y el de scikit-learn tienen el mismo número de hojas y hacen exactamente las mismas predicciones en las cinco profundidades, de la 1 a la 5. Con esto confirmamos que `DecisionTreeClassifier` hace lo que calculamos: probar todos los cortes, elegir el de mayor ganancia y repetir en cada grupo.

### 5.5 Lo que guarda scikit-learn por dentro

Un árbol ya entrenado guarda, para cada nodo, la pregunta, la impureza y cuántos pasajeros de cada clase llegaron. Se consulta en `arbol.tree_`. Veamos un árbol de profundidad 3 como tabla y como texto.

In [ ]:
arbol_chico = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_train_proc, y_train)
estructura = arbol_chico.tree_
nombres = list(X_train_proc.columns)

filas = []
for nodo in range(estructura.node_count):
    es_hoja = estructura.children_left[nodo] == -1
    valores = estructura.value[nodo][0]
    proporcion_no, proporcion_si = valores / valores.sum()   # así no depende de si scikit-learn guarda cantidades o proporciones
    filas.append({
        "nodo": nodo,
        "pregunta": "hoja" if es_hoja else f"{nombres[estructura.feature[nodo]]} <= {estructura.threshold[nodo]:.2f}",
        "si va al nodo": "" if es_hoja else estructura.children_left[nodo],
        "no va al nodo": "" if es_hoja else estructura.children_right[nodo],
        "gini": estructura.impurity[nodo],
        "pasajeros": estructura.n_node_samples[nodo],
        "proporción que no sobrevive": proporcion_no,
        "proporción que sobrevive": proporcion_si,
    })

pd.DataFrame(filas).set_index("nodo").round(3)

In [ ]:
print(export_text(arbol_chico, feature_names=nombres, show_weights=True))

En la tabla, `si va al nodo` indica a dónde se mueve un pasajero que cumple la pregunta y `no va al nodo` a dónde se mueve el que no la cumple. Las dos últimas columnas son la proporción de cada clase en el nodo, y `export_text` con `show_weights=True` muestra las mismas cantidades como número de pasajeros.

Ahora seguimos a un pasajero de test por el árbol, pregunta por pregunta, y comprobamos que llegamos a la misma probabilidad que entrega `predict_proba`.

In [ ]:
pasajero = X_test.iloc[[0]]
pasajero_proc = preprocesador.transform(pasajero)
display(pasajero)

nodos_recorridos = arbol_chico.decision_path(pasajero_proc).indices
for nodo in nodos_recorridos:
    if estructura.children_left[nodo] == -1:
        valores = estructura.value[nodo][0]
        print(f"Nodo {nodo}: hoja. Proporción de sobrevivientes = {valores[1] / valores.sum():.3f}")
    else:
        columna = nombres[estructura.feature[nodo]]
        valor = pasajero_proc[columna].iloc[0]
        cumple = valor <= estructura.threshold[nodo]
        print(f"Nodo {nodo}: {columna} = {valor:.2f} <= {estructura.threshold[nodo]:.2f}? {'sí' if cumple else 'no'}")

print("\npredict_proba de scikit-learn:", arbol_chico.predict_proba(pasajero_proc).round(3))
print("Clase real del pasajero:", y_test.iloc[0])

El pasajero del ejemplo es un hombre de 24 años, de tercera clase, con 3 personas en su grupo y 8.05 libras por persona. Como no es mujer ni niño, cumple la primera condición (`es_mujer_o_nino <= 0.5`) y baja por la izquierda. Pagó 26.14 libras por persona o menos, y más de 3.88, así que llega a una hoja donde sobrevive el 10.3% de los pasajeros (33 de 321). Es la misma probabilidad que entrega `predict_proba` (0.103). Predecir con un árbol es seguir las preguntas hasta una hoja y devolver la proporción de esa hoja.

## 6. Modelo base

Entrenamos el árbol con los valores por defecto, que no le ponen ningún límite. Todavía no usamos el test: la evaluación en test queda para el final, cuando ya hayamos elegido el modelo.

También calculamos tres referencias sencillas para saber si el árbol aporta algo: decir siempre que no sobrevive, la regla "sobrevive si es mujer" y la regla "sobrevive si es mujer o niño".

In [ ]:
modelo_base = armar_modelo().fit(X_train, y_train)
arbol_base = modelo_base.named_steps["arbol"]
scores_base = cross_val_score(modelo_base, X_train, y_train, cv=cv, scoring="accuracy")

print("Profundidad:", arbol_base.get_depth(), "| Hojas:", arbol_base.get_n_leaves())
print("Accuracy en train:", round(accuracy_score(y_train, modelo_base.predict(X_train)), 3))
print(f"Accuracy en validación cruzada: {scores_base.mean():.3f} ± {scores_base.std():.3f}")

print("\nReferencias en train (no tienen nada que aprender, así que train es una medida justa):")
print("Siempre 'no sobrevive':", round((y_train == 0).mean(), 3))
print("Sobrevive si es mujer:  ", round(((X_train["sex"] == "female").astype(int) == y_train).mean(), 3))
print("Sobrevive si es mujer o niño:", round((X_train["es_mujer_o_nino"] == y_train).mean(), 3))

El árbol sin límites llega a 19 niveles y 126 hojas. Acierta 92.8% en train, pero en validación cruzada 81.7%: memoriza, aunque menos que con otras variables. Las referencias, que no aprenden nada: la regla "sobrevive si es mujer" acierta 78.9% en train y la regla "sobrevive si es mujer o niño" 79.5%. Un árbol de 126 hojas supera a esa regla por unos 2 puntos.

Una sola validación cruzada también puede variar. Este mismo árbol dio 0.817 ± 0.021 aquí y 0.820 en la sección 4, donde usamos la validación repetida (50 evaluaciones). Por eso, para decidir entre modelos, usamos la repetida.

## 7. Qué hace cada hiperparámetro

Antes de usar `GridSearchCV`, miramos por separado qué pasa cuando cambiamos `max_depth` (cuántas preguntas seguidas puede hacer) y `min_samples_leaf` (mínimo de pasajeros por hoja). Para cada valor calculamos la accuracy en train y en validación cruzada.

In [ ]:
def curva(parametro, valores):
    filas = []
    for valor in valores:
        res = cross_validate(armar_modelo(**{parametro: valor}), X_train, y_train, cv=cv,
                             scoring="accuracy", return_train_score=True)
        filas.append({parametro: valor, "train": res["train_score"].mean(), "validación": res["test_score"].mean()})
    return pd.DataFrame(filas).set_index(parametro)


curva_profundidad = curva("max_depth", range(1, 16))
curva_hoja = curva("min_samples_leaf", [1, 2, 3, 5, 8, 10, 15, 20, 30, 40, 60])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for tabla, ax, titulo in [(curva_profundidad, axes[0], "max_depth"), (curva_hoja, axes[1], "min_samples_leaf")]:
    ax.plot(tabla.index, tabla["train"], marker="o", label="Train")
    ax.plot(tabla.index, tabla["validación"], marker="o", label="Validación cruzada")
    ax.set_title(f"Accuracy según {titulo}")
    ax.set_xlabel(titulo)
    ax.set_ylabel("Accuracy")
    ax.legend()
plt.tight_layout()
plt.show()

display(curva_profundidad.round(3).T)
display(curva_hoja.round(3).T)

- `max_depth`: la accuracy en train sube sin parar (de 0.795 a 0.931), pero la de validación sube de 0.795 (profundidad 1) a 0.832 con profundidad 3 y 4, y después oscila entre 0.816 y 0.834 sin una tendencia clara. Con 4 variables, un árbol de 3 o 4 niveles ya alcanza casi todo lo que se puede. Con profundidad 1 o 2 el árbol es demasiado simple (0.795 y 0.794).
- `min_samples_leaf`: con hojas de 1 pasajero, train llega a 0.932 y validación se queda en 0.817. La validación sube a 0.841 con hojas de 3 pasajeros y 0.837 con 5, y se mantiene entre 0.817 y 0.833 con hojas de 8 a 20. Con hojas de 60 pasajeros baja a 0.794, porque el árbol ya es demasiado simple.

Las curvas son irregulares y diferencias de un punto quedan dentro del ruido (la desviación entre partes es de unos 0.03). Lo que sí se ve es que sin ningún freno el árbol sobreajusta, y que casi cualquier freno razonable lo arregla.

## 8. Búsqueda de hiperparámetros con GridSearchCV

Ahora probamos todas las combinaciones de una rejilla, con validación cruzada sobre train. Usamos la misma rejilla de la clase, con valores más amplios para `max_depth` y `min_samples_leaf`.

In [ ]:
rejilla = {
    "arbol__criterion": ["gini", "entropy"],
    "arbol__max_depth": [2, 3, 4, 5, 6, 8, None],
    "arbol__min_samples_split": [2, 5, 10],
    "arbol__min_samples_leaf": [1, 2, 4, 8, 16],
}

busqueda = GridSearchCV(armar_modelo(), rejilla, cv=cv, scoring="accuracy")
busqueda.fit(X_train, y_train)

n_combinaciones = len(busqueda.cv_results_["params"])
print(f"Combinaciones probadas: {n_combinaciones} ({n_combinaciones * cv.get_n_splits()} entrenamientos)")
print("Mejores hiperparámetros:", busqueda.best_params_)
print(f"Mejor accuracy en validación cruzada: {busqueda.best_score_:.3f}")

modelo_rejilla = busqueda.best_estimator_
arbol_rejilla = modelo_rejilla.named_steps["arbol"]
print("Profundidad:", arbol_rejilla.get_depth(), "| Hojas:", arbol_rejilla.get_n_leaves())

In [ ]:
resultados = pd.DataFrame(busqueda.cv_results_)
columnas = ["param_arbol__criterion", "param_arbol__max_depth", "param_arbol__min_samples_split",
            "param_arbol__min_samples_leaf", "mean_test_score", "std_test_score", "rank_test_score"]
resultados[columnas].sort_values("rank_test_score", kind="stable").head(10).round(3)

In [ ]:
mejor = busqueda.best_params_
filtro = ((resultados["param_arbol__criterion"] == mejor["arbol__criterion"])
          & (resultados["param_arbol__min_samples_split"] == mejor["arbol__min_samples_split"]))

tabla_calor = (resultados[filtro]
               .assign(max_depth=lambda d: d["param_arbol__max_depth"].map(lambda v: "sin límite" if pd.isna(v) else str(int(v))))
               .pivot_table(index="max_depth", columns="param_arbol__min_samples_leaf", values="mean_test_score"))

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(tabla_calor, annot=True, fmt=".3f", cmap="viridis", ax=ax)
ax.set_title("Accuracy media en validación cruzada")
ax.set_xlabel("min_samples_leaf")
ax.set_ylabel("max_depth")
plt.show()

La búsqueda probó 210 combinaciones. La mejor fue `gini`, `max_depth=4`, `min_samples_leaf=16` y `min_samples_split=2`, con 0.847 en validación cruzada y un árbol de 14 hojas. Hay que mirarla con cuidado:

- Hay muchos empates. Seis combinaciones dan 0.847: cambian el criterio o `min_samples_split`, que no influye porque con hojas de 16 pasajeros un nodo necesita al menos 32 para dividirse. Otras cuatro dan 0.846. Las diferencias son de una milésima, mucho menores que la desviación entre partes (0.028).
- El 0.847 es optimista: es la mejor de 210 combinaciones medida sobre los mismos folds. Más adelante lo medimos otra vez con la validación repetida.
- El mapa de calor muestra que las mejores celdas están en profundidad 3 y 4, con hojas de 8 a 16 pasajeros (de 0.841 a 0.847). En profundidad 5 o más las celdas quedan entre 0.816 y 0.841.

## 9. Poda por costo-complejidad

`max_depth` y `min_samples_leaf` frenan el árbol mientras crece. La poda hace lo contrario: deja crecer el árbol completo y después recorta las ramas que aportan poco. El parámetro `ccp_alpha` es el precio que se cobra por cada hoja: un valor más alto recorta más.

`cost_complexity_pruning_path` calcula los valores de `ccp_alpha` en los que el árbol cambia de tamaño. Para cada uno medimos cuántas hojas quedan y la accuracy en validación cruzada.

In [ ]:
camino = DecisionTreeClassifier(random_state=SEED).cost_complexity_pruning_path(X_train_proc, y_train)
alfas = np.unique(np.round(camino.ccp_alphas[:-1], 5))   # el último valor recorta todo hasta la raíz
print("Valores de ccp_alpha a probar:", len(alfas))

filas = []
for alfa in alfas:
    modelo = armar_modelo(ccp_alpha=alfa)
    scores = cross_val_score(modelo, X_train, y_train, cv=cv, scoring="accuracy")
    hojas = modelo.fit(X_train, y_train).named_steps["arbol"].get_n_leaves()
    filas.append({"ccp_alpha": alfa, "hojas": hojas, "validación": scores.mean()})
poda = pd.DataFrame(filas)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(poda["ccp_alpha"], poda["hojas"], marker="o")
axes[0].set_title("Hojas que quedan según ccp_alpha")
axes[0].set_xlabel("ccp_alpha")
axes[0].set_ylabel("Hojas")
axes[1].plot(poda["ccp_alpha"], poda["validación"], marker="o")
axes[1].set_title("Accuracy en validación cruzada según ccp_alpha")
axes[1].set_xlabel("ccp_alpha")
axes[1].set_ylabel("Accuracy")
plt.tight_layout()
plt.show()

poda.sort_values("validación", ascending=False, kind="stable").head(8).round(4)

In [ ]:
# Entre los alfas que empatan en accuracy, nos quedamos con el más grande: el árbol más pequeño
mejor_validacion = poda["validación"].max()
alfa_elegido = poda[poda["validación"] == mejor_validacion]["ccp_alpha"].max()
modelo_podado = armar_modelo(ccp_alpha=alfa_elegido).fit(X_train, y_train)
arbol_podado = modelo_podado.named_steps["arbol"]

print("ccp_alpha elegido:", alfa_elegido)
print("Profundidad:", arbol_podado.get_depth(), "| Hojas:", arbol_podado.get_n_leaves())
print(f"Accuracy en validación cruzada: {mejor_validacion:.3f}")

La accuracy en validación sube cuando se poda un poco y llega a 0.834 con `ccp_alpha` cerca de 0.0019, que deja un árbol de 26 hojas. La curva es muy plana: árboles de solo 6 hojas (con `ccp_alpha` cerca de 0.0057) dan 0.832. Elegimos el alfa de mayor accuracy y, si hay empate, el más grande, que es el árbol más pequeño. Queda un árbol de profundidad 10 y 26 hojas, bastante más grande que el de la rejilla.

### 9.1 Poda junto con `min_samples_leaf`

Esta combinación se agregó porque, en una versión anterior de este trabajo, el árbol final tenía hojas con muy pocos pasajeros. Para no usar el test en la decisión, se evalúa solo con validación cruzada en train. Para cada valor de `min_samples_leaf` calculamos su propio camino de poda, elegimos su mejor `ccp_alpha` con la validación de 5 partes (el más grande si hay empate) y medimos el resultado con la validación repetida. Si varios valores quedan a menos de 0.01 del mejor, nos quedamos con el que deja menos hojas, igual que en la sección 10.

In [ ]:
filas = []
for hoja in [1, 2, 4, 8, 10, 15, 20, 30]:
    camino_hoja = DecisionTreeClassifier(random_state=SEED, min_samples_leaf=hoja).cost_complexity_pruning_path(X_train_proc, y_train)
    alfas_hoja = np.unique(np.round(camino_hoja.ccp_alphas[:-1], 5))
    validaciones = np.array([
        cross_val_score(armar_modelo(min_samples_leaf=hoja, ccp_alpha=a), X_train, y_train, cv=cv, scoring="accuracy").mean()
        for a in alfas_hoja
    ])
    mejor_alfa_hoja = alfas_hoja[validaciones == validaciones.max()].max()

    modelo_hoja = armar_modelo(min_samples_leaf=hoja, ccp_alpha=mejor_alfa_hoja).fit(X_train, y_train)
    repetida = cross_val_score(modelo_hoja, X_train, y_train, cv=cv_repetida, scoring="accuracy")
    filas.append({
        "min_samples_leaf": hoja,
        "mejor ccp_alpha": mejor_alfa_hoja,
        "profundidad": modelo_hoja.named_steps["arbol"].get_depth(),
        "hojas": modelo_hoja.named_steps["arbol"].get_n_leaves(),
        "validación (5 partes)": validaciones.max(),
        "validación repetida": repetida.mean(),
    })

tabla_hojas_poda = pd.DataFrame(filas).set_index("min_samples_leaf")
display(tabla_hojas_poda.round(4))

mejor_repetida = tabla_hojas_poda["validación repetida"].max()
aceptables_poda = tabla_hojas_poda[tabla_hojas_poda["validación repetida"] >= mejor_repetida - 0.01]
hoja_elegida = int(aceptables_poda["hojas"].idxmin())
alfa_hoja_elegido = float(aceptables_poda.loc[hoja_elegida, "mejor ccp_alpha"])
modelo_podado_hoja = armar_modelo(min_samples_leaf=hoja_elegida, ccp_alpha=alfa_hoja_elegido).fit(X_train, y_train)

print("Valores de min_samples_leaf a menos de 0.01 del mejor:", list(aceptables_poda.index))
print("Elegido: min_samples_leaf =", hoja_elegida, "| ccp_alpha =", alfa_hoja_elegido,
      "| hojas =", modelo_podado_hoja.named_steps["arbol"].get_n_leaves())

Los resultados son parecidos entre sí: la validación repetida va de 0.829 a 0.840. Los mejores son los de hojas de 15 y 20 pasajeros (0.840, con 8 y 7 hojas), y los más bajos los de hojas de 1 y 2 pasajeros (0.829, con 26 y 31 hojas). Con hojas de 8 y 10 pasajeros la poda deja solo 6 hojas y da 0.832.

Por la regla (a menos de 0.01 del mejor, el que tiene menos hojas) queda `min_samples_leaf=8` con `ccp_alpha=0.0069`: un árbol de profundidad 3 y 6 hojas. Es el candidato "podado con hojas mínimas" de la sección 10.

## 10. Elegir el modelo final

Comparamos los cuatro candidatos (base, mejor de la rejilla, podado y podado con hojas mínimas) con validación cruzada repetida, solo con train. La regla para decidir se escribe antes de mirar el resultado: si varios modelos quedan a menos de 0.01 de accuracy del mejor, nos quedamos con el que tiene menos hojas, porque un árbol más pequeño se puede leer y suele generalizar mejor.

In [ ]:
candidatos = {"Base": modelo_base, "Rejilla": modelo_rejilla, "Podado": modelo_podado,
              "Podado + hojas mínimas": modelo_podado_hoja}

filas = []
for nombre, modelo in candidatos.items():
    res = cross_validate(modelo, X_train, y_train, cv=cv_repetida, scoring="accuracy", return_train_score=True)
    arbol = modelo.named_steps["arbol"]
    filas.append({
        "modelo": nombre,
        "profundidad": arbol.get_depth(),
        "hojas": arbol.get_n_leaves(),
        "accuracy train": res["train_score"].mean(),
        "accuracy validación": res["test_score"].mean(),
        "desviación": res["test_score"].std(),
    })
tabla_candidatos = pd.DataFrame(filas).set_index("modelo")
display(tabla_candidatos.round(3))

MARGEN = 0.01
mejor_media = tabla_candidatos["accuracy validación"].max()
aceptables = tabla_candidatos[tabla_candidatos["accuracy validación"] >= mejor_media - MARGEN]
nombre_final = aceptables["hojas"].idxmin()
modelo_final = candidatos[nombre_final]

print("Modelos dentro del margen de", MARGEN, ":", list(aceptables.index))
print("Modelo final:", nombre_final)

Con la validación repetida, el árbol de la rejilla (14 hojas) tiene el mejor resultado, 0.844, y es el único a menos de 0.01 del mejor. El podado (26 hojas) queda en 0.829, el podado con hojas mínimas (6 hojas) en 0.832 y el base (126 hojas) en 0.820. Por la regla que escribimos antes, el modelo final es el **de la rejilla**: `gini`, profundidad 4, `min_samples_leaf=16` y 14 hojas.

Dos cosas para tener presentes:

- El 0.844 de la rejilla viene de haber elegido el mejor de 210 combinaciones, así que es algo optimista. El podado con 6 hojas, más simple, queda a solo 0.012 (0.832).
- El hueco entre train y validación es pequeño en el de la rejilla (0.852 contra 0.844) y grande en el base (0.931 contra 0.820).

## 11. Evaluación en test

Esta es la primera vez que usamos el test. Comparamos las referencias sencillas con los cuatro candidatos. La decisión del modelo final ya está tomada, así que estos números no la cambian.

In [ ]:
y_test_np = y_test.to_numpy()

modelos_a_comparar = [
    ("Siempre 'no sobrevive'", np.zeros(len(y_test_np), dtype=int), None),
    ("Sobrevive si es mujer", (X_test["sex"] == "female").astype(int).to_numpy(), None),
    ("Sobrevive si es mujer o niño", X_test["es_mujer_o_nino"].to_numpy(), None),
]
for nombre, modelo in candidatos.items():
    modelos_a_comparar.append((f"Árbol {nombre.lower()}", modelo.predict(X_test), modelo))

filas = []
for nombre, pred, modelo in modelos_a_comparar:
    filas.append({
        "modelo": nombre,
        f"aciertos (de {len(y_test_np)})": int((pred == y_test_np).sum()),
        "accuracy test": (pred == y_test_np).mean(),
        "AUC test": np.nan if modelo is None else roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1]),
    })

pd.DataFrame(filas).set_index("modelo").round(3)

Primero una advertencia: el test tiene 179 pasajeros, así que cada pasajero vale 0.56 puntos de accuracy. Con eso en mente:

- Los cuatro árboles quedan entre 0.804 y 0.816 (de 144 a 146 aciertos) y las dos reglas sencillas en 0.777 y 0.782 (139 y 140). Los árboles ganan por 4 a 7 pasajeros: una ventaja, pero chica para un test de este tamaño.
- El modelo final (la rejilla) acierta 144 de 179, y el árbol base, sin límites, acierta 146. En test el orden entre los árboles no sigue el de la validación cruzada, pero la diferencia es de dos pasajeros y un test de este tamaño no la puede distinguir. Por eso decidimos con la validación y no con el test.
- En AUC, la rejilla es la mejor (0.866, contra 0.836 a 0.845 de los demás). Un árbol con hojas mezcladas da probabilidades más útiles que uno con hojas puras. Con 179 pasajeros, estas diferencias también pueden ser ruido.

In [ ]:
# Error estándar de una proporción: raíz de p(1 - p) / n
accuracy_final = accuracy_score(y_test, modelo_final.predict(X_test))
error_estandar = np.sqrt(accuracy_final * (1 - accuracy_final) / len(y_test))
print(f"Accuracy del modelo final en test: {accuracy_final:.3f}")
print(f"Error estándar con {len(y_test)} pasajeros: {error_estandar:.3f}")
print(f"Rango razonable (± 2 errores estándar): {accuracy_final - 2 * error_estandar:.2f} a {accuracy_final + 2 * error_estandar:.2f}")
print("Puntos de accuracy que vale un solo pasajero:", round(100 / len(y_test), 2))

El error estándar con 179 pasajeros es de unos 3 puntos, así que una accuracy de 0.80 se lee como "entre 0.75 y 0.86". Esa incertidumbre es más grande que las diferencias entre los árboles, y es la razón por la que nos fiamos más de la validación cruzada que de un solo test para comparar. En la sección 14 vemos qué pasa con otros repartos.

In [ ]:
pred_final = modelo_final.predict(X_test)

matriz = confusion_matrix(y_test, pred_final)
fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["No", "Sí"], yticklabels=["No", "Sí"], ax=ax)
ax.set_title(f"Matriz de confusión del árbol {nombre_final.lower()}")
ax.set_xlabel("Predicha")
ax.set_ylabel("Real")
plt.show()

print(confusion_matrix(y_test, pred_final))
print(classification_report(y_test, pred_final, target_names=["No sobrevivió", "Sobrevivió"], digits=3))

El modelo final acierta 93 de los 110 pasajeros que no sobrevivieron (recall 0.845, el porcentaje de los reales que el modelo encontró) y 51 de los 69 que sí sobrevivieron (recall 0.739). Se equivoca casi igual al decir que alguien sobrevive cuando no (17 casos) que al contrario (18 casos).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for nombre, modelo in candidatos.items():
    probabilidades = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probabilidades)
    ax.plot(fpr, tpr, label=f"{nombre} (AUC = {roc_auc_score(y_test, probabilidades):.3f})")
ax.plot([0, 1], [0, 1], "k--", label="Azar")
ax.set_title("Curva ROC en test")
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Tasa de verdaderos positivos")
ax.legend(loc="lower right")
plt.show()

Las curvas de los cuatro árboles quedan cerca entre sí y la de la rejilla va por arriba en buena parte del recorrido. El árbol base tiene hojas casi puras, así que casi todas sus probabilidades son 0 o 1 y su curva tiene menos puntos intermedios.

In [ ]:
tamanos, scores_train, scores_val = learning_curve(
    modelo_final, X_train, y_train, cv=cv, scoring="accuracy",
    train_sizes=np.linspace(0.1, 1.0, 8), shuffle=True, random_state=SEED,
)

fig, ax = plt.subplots(figsize=(7, 4.5))
for scores, etiqueta in [(scores_train, "Train"), (scores_val, "Validación")]:
    ax.plot(tamanos, scores.mean(axis=1), marker="o", label=etiqueta)
    ax.fill_between(tamanos, scores.mean(axis=1) - scores.std(axis=1), scores.mean(axis=1) + scores.std(axis=1), alpha=0.15)
ax.set_title(f"Curva de aprendizaje del árbol {nombre_final.lower()}")
ax.set_xlabel("Pasajeros de entrenamiento")
ax.set_ylabel("Accuracy")
ax.legend()
plt.show()

Con pocos pasajeros (unos 60) el árbol valida mal, cerca de 0.77, y train está en 0.78. Con unos 200 pasajeros la validación se acerca a 0.82, luego se queda plana y vuelve a subir hasta 0.847 al final, casi pegada a train (0.852). Las dos curvas terminan juntas, así que el árbol no está sobreajustando. La validación todavía sube al final, por lo que más pasajeros podrían ayudar un poco. La sombra ancha, que es la desviación entre partes, recuerda que con tan pocos datos cada resultado se mueve.

## 12. ¿Dónde se equivoca el modelo?

Un accuracy global esconde en qué grupos falla. Calculamos el porcentaje de aciertos en test por sexo y clase.

In [ ]:
errores = X_test[["sex", "pclass"]].copy()
errores["predicho"] = pred_final
errores["sobrevivió"] = y_test_np
errores["acierto"] = errores["predicho"] == errores["sobrevivió"]

por_grupo = errores.groupby(["sex", "pclass"]).agg(
    pasajeros=("acierto", "size"),
    sobrevivieron=("sobrevivió", "sum"),
    aciertos_pct=("acierto", lambda s: s.mean() * 100),
)
display(por_grupo.round(1))

# En los dos grupos con más errores: qué dijo el modelo y qué pasó en realidad
for sexo, clase in [("male", 1), ("female", 3)]:
    grupo = errores[(errores["sex"] == sexo) & (errores["pclass"] == clase)]
    print(f"sex = {sexo}, pclass = {clase}")
    display(pd.crosstab(grupo["sobrevivió"], grupo["predicho"], rownames=["real"], colnames=["predicho"]))

El modelo acierta más en las mujeres de primera y segunda clase (93.3% y 94.4%) y en los hombres de segunda y tercera (87.5% y 90.3%). Donde más falla:

- **Hombres de primera clase: 53.3% de aciertos.** Sobrevivieron 11 de 30. El modelo dijo "sí sobrevive" a 15 de ellos y solo 6 sobrevivieron. A los otros 15 les dijo "no" y se equivocó con 5. Si hubiera dicho "no" a los 30, habría acertado 19 (63.3%), más que con las preguntas del árbol.
- **Mujeres de tercera clase: 64.3% de aciertos.** Sobrevivieron 14 de 28. El modelo dijo "sí" a 16: acertó con 10 y se equivocó con 6. Dijo "no" a 12: acertó con 8 y se equivocó con 4. Aquí sí supera a decir "sí" a todas, que acertaría 14 de 28 (50%).

Son los dos grupos que comparamos en el proyecto 1 (mujeres de tercera clase contra hombres de primera), donde el sexo y la clase se contradicen. Además son grupos chicos (entre 15 y 72 pasajeros), así que sus porcentajes cambian mucho con pocos casos. En la sección 13 vemos de qué hojas salen estos errores.

## 13. Leer el árbol final

Como no escalamos las variables, los umbrales del árbol están en sus unidades reales: años, libras por persona y número de personas.

In [ ]:
arbol_final = modelo_final.named_steps["arbol"]
nombres_finales = list(modelo_final.named_steps["preprocesador"].get_feature_names_out())

plt.figure(figsize=(20, 9))
plot_tree(arbol_final, feature_names=nombres_finales, class_names=["No", "Sí"],
          filled=True, rounded=True, fontsize=9, node_ids=True)
plt.title(f"Árbol {nombre_final.lower()}")
plt.show()

In [ ]:
print(export_text(arbol_final, feature_names=nombres_finales))

In [ ]:
# Qué pasa en cada hoja, con los pasajeros de train y de test
preprocesador_final = modelo_final.named_steps["preprocesador"]
hojas_train = arbol_final.apply(preprocesador_final.transform(X_train))
hojas_test = arbol_final.apply(preprocesador_final.transform(X_test))

resumen_train = pd.DataFrame({"hoja": hojas_train, "sobrevivió": y_train_np}).groupby("hoja")["sobrevivió"].agg(["size", "mean"])
resumen_test = pd.DataFrame({"hoja": hojas_test, "sobrevivió": y_test_np}).groupby("hoja")["sobrevivió"].agg(["size", "mean"])

tabla_hojas = resumen_train.join(resumen_test, lsuffix="_train", rsuffix="_test")
tabla_hojas.columns = ["pasajeros train", "sobrevive train (%)", "pasajeros test", "sobrevive test (%)"]
tabla_hojas[["sobrevive train (%)", "sobrevive test (%)"]] *= 100
valores_hojas = arbol_final.tree_.value[tabla_hojas.index, 0, :]
tabla_hojas["predice"] = np.where(valores_hojas[:, 1] > valores_hojas[:, 0], "Sí", "No")
tabla_hojas.round(1)

In [ ]:
importancias = pd.Series(arbol_final.feature_importances_, index=nombres_finales).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(x=importancias.values, y=importancias.index, color="tab:blue", ax=ax)
ax.set_title("Importancia de las variables")
ax.set_xlabel("Importancia (reducción total de impureza)")
ax.set_ylabel("")
plt.show()

importancias.round(3)

Las hojas más importantes, con el número de nodo que aparece en la figura:

- **Hojas 17, 18 y 19.** Mujeres y niños de primera o segunda clase: sobreviven. Son las más sólidas: de 88.9% a 100% en train y de 91.7% a 100% en test (de 9 a 14 pasajeros cada una).
- **Hoja 22.** Mujeres y niños de tercera clase en grupos de hasta 4 personas que pagaron 7.89 libras por persona o menos: sobreviven (67.1% en train y 68.4% en test, con 19 pasajeros). Si pagaron más (hoja 23) la hoja queda mezclada (39.1% en train, 50.0% en test con 8) y el árbol dice "no".
- **Hojas 25 y 26.** Mujeres y niños de tercera clase en grupos de 5 o más: no sobreviven (22.2% y 0.0% en train, y 0.0% en test con 8 pasajeros entre las dos).
- **Hoja 6.** Quienes no son mujer ni niño y pagaron entre 6.05 y 26.14 libras por persona: no sobreviven. Es la hoja más grande y más estable: 304 pasajeros en train con 9.5% de supervivencia y 75 en test con 10.7%.
- **Hojas 9 y 10.** Quienes no son mujer ni niño y pagaron entre 26.14 y 30.59 libras por persona: el árbol dice que sobreviven (63.6% en train con 33 pasajeros), pero en test baja a 35.7% con 14 pasajeros (5 sobrevivieron).
- **Hojas 3 y 13.** Tienen solo 2 pasajeros en test cada una: no se pueden evaluar.

Las hojas 9 y 10 son lo menos confiable del árbol. Es una ventana estrecha de tarifa con pocos pasajeros, y parece una casualidad del train y no una regla del barco. En test juntan 14 pasajeros y 9 no sobrevivieron, la misma cuenta de errores que vimos en los hombres de primera clase.

Importancia de las variables: `num__es_mujer_o_nino` pesa 0.611, `num__pclass` 0.177, `num__fare_por_persona` 0.141 y `num__family_size` 0.071. El árbol usa las cuatro, y la norma de abordaje concentra más de la mitad. La importancia solo cuenta cuánto baja la impureza dentro de este árbol de 14 hojas, así que no dice qué tanto influyó cada variable en el hundimiento.

## 14. ¿Qué tan especial es este reparto de train y test?

En el test de este notebook, el modelo final saca 0.804. Es normal preguntarse si eso es alto o bajo, porque otras personas pueden sacar más o menos con un modelo parecido. Parte de la respuesta es que el resultado depende de qué pasajeros caen en test. Para medir cuánto cambia, repetimos el proceso completo con 20 repartos distintos (`random_state` de 0 a 19). En cada uno separamos, elegimos los hiperparámetros solo con el train de ese reparto (con una rejilla más chica, para que sea rápido) y medimos en su test. Esto no sirve para escoger el modelo final; solo mide cuánto varía el resultado.

Comparamos cuatro cosas: la regla "sobrevive si es mujer", la regla "sobrevive si es mujer o niño", un árbol con nuestras 4 variables y un árbol con las columnas originales.

Una limitación: elegimos las variables con el train del reparto 42, que contiene filas que en otros repartos caen en test. Por eso la ventaja de nuestras 4 variables sobre las originales puede estar un poco inflada. La comparación contra las reglas sencillas no tiene ese problema, porque las reglas no aprenden nada.

In [ ]:
rejilla_chica = {
    "arbol__criterion": ["gini", "entropy"],
    "arbol__max_depth": [3, 4, 5, 6],
    "arbol__min_samples_leaf": [1, 2, 4, 8, 16],
}
ARBOL_NUESTRO = "Árbol con nuestras 4 variables"
ARBOL_7 = "Árbol con las columnas originales"
REGLA = "Regla: sobrevive si es mujer"
REGLA_2 = "Regla: sobrevive si es mujer o niño"
variables_a_comparar = {
    ARBOL_NUESTRO: (numericas, nominales),
    ARBOL_7: (["age", "pclass", "sibsp", "parch", "fare"], ["sex", "embarked"]),
}

filas = []
for semilla in list(range(20)) + [SEED]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=semilla)
    fila = {"reparto": semilla, REGLA: accuracy_score(y_te, (X_te["sex"] == "female").astype(int)),
            REGLA_2: accuracy_score(y_te, X_te["es_mujer_o_nino"])}
    for nombre, (num, nom) in variables_a_comparar.items():
        modelo = Pipeline([
            ("preprocesador", armar_preprocesador(num, nom)),
            ("arbol", DecisionTreeClassifier(random_state=SEED)),
        ])
        busqueda_reparto = GridSearchCV(modelo, rejilla_chica, cv=cv, scoring="accuracy").fit(X_tr, y_tr)
        fila[nombre] = accuracy_score(y_te, busqueda_reparto.predict(X_te))
    filas.append(fila)
repartos = pd.DataFrame(filas).set_index("reparto")

otros_repartos = repartos.drop(index=SEED)   # los 20 repartos distintos al del notebook
resumen_repartos = pd.DataFrame({
    "media": otros_repartos.mean(),
    "mínimo": otros_repartos.min(),
    "máximo": otros_repartos.max(),
    "repartos con 0.80 o más (de 20)": (otros_repartos >= 0.80).sum(),
    "reparto 42 (el del notebook)": repartos.loc[SEED],
})
display(resumen_repartos.round(3))
print("Repartos en los que el árbol con nuestras 4 variables supera a la regla de sexo:", (otros_repartos[ARBOL_NUESTRO] > otros_repartos[REGLA]).sum(), "de 20")
print("Repartos en los que empata:", (otros_repartos[ARBOL_NUESTRO] == otros_repartos[REGLA]).sum())
print("Contra la regla de mujer o niño: supera en", (otros_repartos[ARBOL_NUESTRO] > otros_repartos[REGLA_2]).sum(),
      "| empata en", (otros_repartos[ARBOL_NUESTRO] == otros_repartos[REGLA_2]).sum(), "| pierde en", (otros_repartos[ARBOL_NUESTRO] < otros_repartos[REGLA_2]).sum(), "de 20")
print("Diferencia media contra la regla de mujer o niño:", round((otros_repartos[ARBOL_NUESTRO] - otros_repartos[REGLA_2]).mean(), 3))
print("Diferencia media contra la regla:", round((otros_repartos[ARBOL_NUESTRO] - otros_repartos[REGLA]).mean(), 3))

In [ ]:
largo = otros_repartos.reset_index().melt(id_vars="reparto", var_name="modelo", value_name="accuracy en test")

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.stripplot(data=largo, x="accuracy en test", y="modelo", color="tab:blue", size=7, alpha=0.7, ax=ax)
for posicion, modelo in enumerate(otros_repartos.columns):
    ax.plot(repartos.loc[SEED, modelo], posicion, marker="D", color="tab:red", markersize=9,
            label="Reparto 42" if posicion == 0 else None)
ax.axvline(0.80, color="black", linestyle="--", linewidth=1)
ax.set_title("Accuracy en test con 20 repartos distintos (cada punto es un reparto)")
ax.set_xlabel("Accuracy en test")
ax.set_ylabel("")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

Resultados:

- El árbol con nuestras 4 variables saca 0.839 de media (de 0.816 a 0.860) y llega a 0.80 o más en los 20 repartos. Con las columnas originales saca 0.818 de media y llega a 0.80 o más en 16 de 20.
- La regla "sobrevive si es mujer" saca 0.790 de media y llega a 0.80 o más en solo 5 de 20. La regla "sobrevive si es mujer o niño" saca 0.799 y llega en 8 de 20.
- El árbol supera a las dos reglas en los 20 repartos. La diferencia media es de 4.9 puntos contra la regla de sexo y de 4.1 contra la de mujer o niño, parecida a la que vimos en validación cruzada.
- El reparto 42, el de este notebook, queda en la parte baja para el árbol: saca ahí 0.804 con esta rejilla, menos que el mínimo de los otros 20 repartos (0.816). Las reglas sencillas también sacan ahí menos que su promedio (0.777 y 0.782).

Entonces, sacar 0.80 o más en test es lo esperable para un árbol de este tipo en el Titanic, y por sí solo no indica que alguien haya hecho un mejor modelo. Con un solo reparto no se distingue un buen modelo de un reparto favorable. Lo que sí se puede comparar es el promedio en varios repartos y la validación cruzada, que aquí dan alrededor de 0.84.

## Conclusiones

- **Variables.** Partimos de las decisiones del proyecto 1 y comparamos siete variables candidatas nuevas, cada una con una razón del hundimiento. Nos quedamos con 4: `es_mujer_o_nino` (la norma de abordaje de los botes), `pclass`, `fare_por_persona` (lo que pagó cada persona por su boleto) y `family_size` (el tamaño del grupo). `es_mujer_o_nino` reemplaza a `age` y `sex` sin perder nada y deja un árbol que sobreajusta mucho menos. `family_size` se conserva porque quitarla baja el resultado y porque funciona mejor que las alternativas más simples (`viaja_solo` y `familia_pequena`). `cubierta_conocida` parecía fuerte, pero es casi una copia de la clase.
- **Cómo decide un árbol.** En cada nodo prueba todas las preguntas "variable <= umbral", calcula cuánto baja la impureza (Gini) y se queda con la de mayor ganancia. Lo calculamos a mano y obtuvimos los mismos árboles que scikit-learn, con 100% de predicciones iguales en las profundidades de 1 a 5.
- **Sobreajuste.** Un árbol sin límites tiene 126 hojas, acierta 92.8% en train y 81.7% en validación (0.820 en la repetida). Limitarlo (profundidad 4 y hojas de al menos 16 pasajeros) sube la validación a 0.844 con solo 14 hojas.
- **Modelo final.** El árbol de la rejilla: `gini`, profundidad 4, `min_samples_leaf=16` y 14 hojas. Saca 0.844 en validación repetida y, en test, 0.804 de accuracy (144 de 179) y 0.866 de AUC. La poda y la poda con hojas mínimas dieron 0.829 y 0.832, un poco peores, y no cambiaron la decisión (secciones 9 y 9.1).
- **Lo que sí y lo que no podemos afirmar.** Con 179 pasajeros en test el error estándar es de unos 3 puntos, y en este reparto la ventaja sobre las reglas sencillas es de solo 4 a 7 pasajeros. Con 20 repartos distintos sí se ve: el árbol saca 0.839 de media contra 0.790 y 0.799 de las reglas sencillas, y las supera en los 20. Sacar 0.80 o más en test es lo esperable, y el reparto 42 es de los bajos.
- **Dónde falla.** En los hombres de primera clase y en las mujeres de tercera, los grupos donde el sexo y la clase se contradicen. Parte del error sale de las hojas 9 y 10, una ventana de tarifa estrecha (de 26.14 a 30.59 libras por persona) cuyo 63.6% de supervivencia en train baja a 35.7% en test.
- **Qué probaríamos después.** Medir todo el proceso, incluida la elección de variables, con validación cruzada anidada, que la clase deja para más adelante. También ayudaría tener el nombre de cada pasajero, porque su título (Mr, Mrs, Miss, Master) suele ser una variable muy buena y este conjunto no lo trae, y el número de boleto, para saber el tamaño real del grupo.

### Decisiones

| # | Decisión | Opciones | Elección | Evidencia (solo train) |
|---|---|---|---|---|
| 1 | Variables de entrada | Las 7 originales (A) o un conjunto elegido | `es_mujer_o_nino`, `pclass`, `fare_por_persona`, `family_size` | 0.841 contra 0.815 con profundidad 4, y 0.828 contra 0.809 con profundidad 5 |
| 2 | Regla de abordaje | `age` y `sex` por separado o `es_mujer_o_nino` | `es_mujer_o_nino` | Promedio de tres árboles: 0.830 contra 0.808 con `age` y `sex` |
| 3 | `family_size` | Conservarla, quitarla o cambiarla por `viaja_solo` | Conservarla | Sin ella 0.817 y 0.823; con `viaja_solo` 0.815 y 0.821 |
| 4 | Precio | `fare` o `fare_por_persona` | `fare_por_persona` | 0.829 contra 0.812 |
| 5 | `cubierta_conocida` | Agregarla o no | No agregarla | 0.822 contra 0.829, y casi copia la clase |
| 6 | `edad_faltante`, `age_group`, `who`, `embarked` | Agregarlas o no | No agregarlas | Entre 0.824 y 0.834 contra 0.829, dentro del ruido |
| 7 | Escalado y logaritmo | Aplicarlos o no | No aplicarlos | Resultados idénticos con 3 decimales |
| 8 | Faltantes | Rellenar antes o dentro del Pipeline | Dentro del Pipeline (mediana de cada parte) | Evita usar las filas que se están evaluando |
| 9 | Control del sobreajuste | Sin límites, rejilla, poda o poda con hojas mínimas | Rejilla | 0.820, 0.844, 0.829 y 0.832 en validación repetida |
| 10 | Modelo final | Entre los cuatro candidatos | Árbol de la rejilla (14 hojas) | Regla: a menos de 0.01 del mejor, el de menos hojas (era el único) |